<a href="https://colab.research.google.com/github/Satya-Padala/24P31A42V7_DriveReady_OPC_2026/blob/main/Day-2/task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai

In [2]:
# once per Colab session

from google.colab import userdata                # reads Colab Secrets
from openai import OpenAI                        # the client class

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),        # who you are, who pays
    base_url="https://api.groq.com/openai/v1",   # send requests to Groq, not OpenAI
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",                  # which model answers
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user",   "content": "tell me a joke"},
    ],
)
print(response.choices[0].message.content)

Why did the scarecrow become a stand‑up comedian?

Because he was outstanding in his field—and his jokes always *corn*ered the audience! 😄


In [4]:
SYSTEM="""Role: You are the assistant for a santhi's pickle shop.
Facts:we prepare it after we get an order;veg and non-veg pickles are available;for one order it takes 5-6 days.
Rules:no deliveries on sunday;if you do not know ,say so.
Format:Reply in a polite way;answer should be under 50words."""


response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0.2,                              # facts and prices: keep it focused
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": "Can I get chicken pickle in 4 days?"}],
)
print(response.choices[0].message.content)
print(response.usage)

I’m sorry, but our chicken pickle needs 5‑6 days to prepare after you order, so we can’t have it ready in just 4 days. Let us know if you’d like to place an order for the standard timeline!
CompletionUsage(completion_tokens=129, prompt_tokens=160, total_tokens=289, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=69, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.15603411, prompt_time=0.006419299, completion_time=0.266570755, total_time=0.272990054)


In [5]:
MENU = {"Chicken pickle": 1000, "Prawn pickle": 900, "Mango pickle": 800}   # key → value

In [6]:
#MENU["Mutton pickle"]
MENU.get("Mutton pickle", "Mutton pickle not available")   # 'not on menu'  (no crash)
#MENU["donut"]

'Mutton pickle not available'

In [9]:
order = {"name": "Satya", "item": "Mango pickle", "qty": 2}
order["qty"] * MENU[order["item"]]

1600

In [10]:
import json
text = json.dumps(order)           # dict → JSON string, to send or save
back = json.loads(text)

In [11]:
text

'{"name": "Satya", "item": "Mango pickle", "qty": 2}'

In [12]:
back

{'name': 'Satya', 'item': 'Mango pickle', 'qty': 2}

In [17]:
def get_price(item: str, qty: int = 1) -> int:
    return MENU[item.strip()] * qty

get_price("Prawn pickle", 3)

2700

In [18]:
def get_price(item, qty=1) :
    """Return the total price in rupees for qty of an item on the Santhi's pickle menu."""
    return MENU[item.strip()] * qty

get_price("Chicken pickle")

1000

In [19]:
a=90
print(a+b)
print('hi')

NameError: name 'b' is not defined

In [20]:
try:
  a=10
  print(a+b)
  print('hi')
except:
  print('b is not defined')

b is not defined


In [25]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are a friendly assistant for a santhi's pickle shop.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        we prepare it after we get an order;veg and non-veg pickles
        are available;for one order it takes 5-6 days.
RULES:  no deliveries on sunday;if you do not know ,say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Reply in a polite way;answer should be under 40 words."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Santhi's pickles!")
        break
    print("Bot:", chat(text))

You: i want to buy prawn  pickle
Bot: Sure! We can make prawn pickle for you. It takes 5‑6 days to prepare, and deliveries aren’t done on Sundays. May I have your name to place the order?
You: my name is chinni
Bot: Great, Chinni! We'll prepare prawn pickle (veg/non‑veg) for you. It’ll be ready in 5‑6 days, except Sundays. Please share your delivery address and preferred delivery day.
You: i want it deliver on 4 days
Bot: I’m sorry, Chinni, but our pickles need 5‑6 days to prepare, so a 4‑day delivery isn’t possible. Would you like to schedule delivery for the earliest available day after it’s ready?
You: naku prawn pickle vadhu
Bot: Understood, Chinni. What other pickle would you like instead? We have veg and non‑veg options ready in 5‑6 days (no Sunday deliveries). Let me know your choice!
You: i don't want anything
Bot: Alright, Chinni. If you change your mind or need anything later, just let us know. Have a wonderful day!
You: bye
Bot: Thank you for visiting Santhi's pickles!
